# Data Visualization with Python: From Static to Interactive

A comprehensive guide covering Matplotlib, Seaborn, and Plotly for exploratory data analysis, statistical visualization, and communicating insights effectively.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('All libraries loaded successfully.')

---
## 1. Why Visualization Matters: Anscombe's Quartet

Anscombe's Quartet demonstrates why summary statistics alone are insufficient. Four datasets with nearly identical means, variances, and correlations look completely different when plotted.

In [ ]:
anscombe = sns.load_dataset('anscombe')
print(anscombe.groupby('dataset').describe().round(2))

In [ ]:
datasets = anscombe['dataset'].unique()
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, ds in zip(axes.flat, datasets):
    subset = anscombe[anscombe['dataset'] == ds]
    ax.scatter(subset['x'], subset['y'], s=50, color='#2c7bb6', edgecolors='white', linewidth=0.5)
    
    m, b = np.polyfit(subset['x'], subset['y'], 1)
    x_line = np.linspace(subset['x'].min(), subset['x'].max(), 100)
    ax.plot(x_line, m * x_line + b, color='#d7191c', linestyle='--', linewidth=1.5)
    
    stats = f"μx={subset['x'].mean():.2f}, μy={subset['y'].mean():.2f}\nr={subset['x'].corr(subset['y']):.3f}"
    ax.set_title(f'Dataset {ds}\n{stats}', fontsize=11)
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)

fig.suptitle("Anscombe's Quartet: Identical Statistics, Very Different Stories", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Matplotlib Architecture: Figure, Axes, Artist

Matplotlib has a hierarchical structure:
- **Figure**: the top-level container (the whole canvas/window)
- **Axes**: the plotting area with data space coordinates (not to be confused with axis)
- **Artist**: everything drawn on the canvas (lines, text, patches, etc.)

Two APIs exist:
1. **pyplot (stateful)**: `plt.plot()`, `plt.xlabel()` — implicit figure/axes management
2. **Object-oriented (explicit)**: `fig, ax = plt.subplots()` followed by `ax.plot()` — recommended for complex work

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.linspace(0, 4 * np.pi, 200)
line1, = ax.plot(x, np.sin(x), label='sin(x)', color='#1b9e77')
line2, = ax.plot(x, np.cos(x), label='cos(x)', color='#d95f02')

ax.set_title('Figure → Axes → Artists', fontsize=13, fontweight='bold')
ax.set_xlabel('x (radians)')
ax.set_ylabel('y')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4 * np.pi)

print(f'Figure type:  {type(fig).__name__}')
print(f'Axes type:    {type(ax).__name__}')
print(f'Line2D type:  {type(line1).__name__}')
print(f'Axes children: {[type(a).__name__ for a in ax.get_children()[:5]]}...')
plt.show()

---
## 3. Basic Plots with Pyplot

In [ ]:
np.random.seed(42)
data = np.random.randn(1000)
categories = ['A', 'B', 'C', 'D', 'E']
values = np.random.randint(10, 50, size=5)
x = np.linspace(0, 10, 100)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].plot(x, np.sin(x) * np.exp(-x / 5), color='#1f77b4', linewidth=2)
axes[0, 0].set_title('Line Plot')

axes[0, 1].scatter(np.random.randn(50), np.random.randn(50), c=np.random.randn(50),
                   cmap='viridis', s=60, alpha=0.7, edgecolors='k', linewidth=0.5)
axes[0, 1].set_title('Scatter Plot')

axes[0, 2].bar(categories, values, color=['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00'])
axes[0, 2].set_title('Bar Chart')

axes[1, 0].hist(data, bins=30, color='#1b9e77', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('Histogram')

axes[1, 1].boxplot([np.random.randn(100), np.random.randn(100) * 1.5 + 1,
                    np.random.randn(100) * 0.8 - 1, np.random.randn(100) * 2],
                   labels=['Group 1', 'Group 2', 'Group 3', 'Group 4'],
                   patch_artist=True,
                   boxprops=dict(facecolor='#fee090', alpha=0.7))
axes[1, 1].set_title('Box Plot')

vp = axes[1, 2].violinplot([np.random.randn(80), np.random.randn(80) * 1.2 + 2],
                           showmeans=True, showmedians=True)
axes[1, 2].set_xticks([1, 2])
axes[1, 2].set_xticklabels(['Control', 'Treatment'])
axes[1, 2].set_title('Violin Plot')
for body in vp['bodies']:
    body.set_facecolor('#7570b3')
    body.set_alpha(0.6)

plt.tight_layout()
plt.show()

---
## 4. Customizing Plots

In [ ]:
np.random.seed(7)

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(1, 13)
sales_a = np.random.randint(80, 150, size=12)
sales_b = np.random.randint(60, 130, size=12)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

ax.plot(x, sales_a, color='#d7191c', marker='o', linestyle='-', linewidth=2,
        markersize=8, markerfacecolor='white', markeredgewidth=2, label='Product A')
ax.plot(x, sales_b, color='#2c7bb6', marker='s', linestyle='--', linewidth=2,
        markersize=8, markerfacecolor='white', markeredgewidth=2, label='Product B')

ax.fill_between(x, sales_a, sales_b, alpha=0.1, color='gray')

ax.set_title('Monthly Sales Comparison (2025)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Revenue (thousands USD)', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(months, rotation=30, ha='right')
ax.legend(frameon=True, shadow=True, fancybox=True, fontsize=10)
ax.grid(True, linestyle=':', alpha=0.5, color='gray')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 180)

for i in range(12):
    diff = abs(sales_a[i] - sales_b[i])
    if diff > 30:
        ax.annotate(f'Δ{diff}', (x[i], (sales_a[i] + sales_b[i]) / 2),
                    textcoords='offset points', xytext=(0, 10), ha='center',
                    fontsize=8, color='#666666',
                    arrowprops=dict(arrowstyle='->', color='#666666', lw=0.5))

plt.tight_layout()
plt.show()

---
## 5. Complex Subplots with GridSpec

In [ ]:
from matplotlib.gridspec import GridSpec

np.random.seed(42)
df_iris = sns.load_dataset('iris')

fig = plt.figure(figsize=(14, 10))
gs = GridSpec(4, 4, figure=fig, hspace=0.3, wspace=0.3)

scatter_ax = fig.add_subplot(gs[0:2, 0:2])
hist_x_ax = fig.add_subplot(gs[0, 2:4])
hist_y_ax = fig.add_subplot(gs[1, 2])
box_ax = fig.add_subplot(gs[2, :])
bar_ax = fig.add_subplot(gs[3, 0:2])
table_ax = fig.add_subplot(gs[3, 2:4])

species_colors = {'setosa': '#e41a1c', 'versicolor': '#4daf4a', 'virginica': '#377eb8'}
for species, color in species_colors.items():
    subset = df_iris[df_iris['species'] == species]
    scatter_ax.scatter(subset['sepal_length'], subset['sepal_width'],
                       c=color, label=species, alpha=0.7, edgecolors='white', s=40)
scatter_ax.set_xlabel('Sepal Length (cm)')
scatter_ax.set_ylabel('Sepal Width (cm)')
scatter_ax.legend(fontsize=8)
scatter_ax.set_title('Sepal Dimensions by Species')

hist_x_ax.hist([df_iris[df_iris['species'] == s]['sepal_length'] for s in species_colors],
               bins=12, color=list(species_colors.values()), label=list(species_colors.keys()),
               alpha=0.6, stacked=True)
hist_x_ax.set_title('Sepal Length Distribution')
hist_x_ax.legend(fontsize=7)

hist_y_ax.hist([df_iris[df_iris['species'] == s]['sepal_width'] for s in species_colors],
               bins=12, color=list(species_colors.values()), alpha=0.6, stacked=True,
               orientation='horizontal')
hist_y_ax.set_title('Sepal Width')

df_iris.boxplot(column='petal_length', by='species', ax=box_ax)
box_ax.set_title('Petal Length by Species')
box_ax.set_xlabel('')

means = df_iris.groupby('species')[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].mean()
means.plot(kind='bar', ax=bar_ax, color=['#e41a1c', '#4daf4a', '#377eb8', '#984ea3'])
bar_ax.set_title('Mean Measurements per Species')
bar_ax.legend(fontsize=7)
bar_ax.set_xticklabels(bar_ax.get_xticklabels(), rotation=0)

table_ax.axis('off')
table_data = [['Metric', 'Setosa', 'Versicolor', 'Virginica']]
for col in ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']:
    row = [col] + [f'{means.loc[s, col]:.2f}' for s in species_colors]
    table_data.append(row)
tbl = table_ax.table(cellText=table_data[1:], colLabels=table_data[0],
                     loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.5)
table_ax.set_title('Summary Statistics', fontsize=10)

fig.suptitle('Iris Dataset: Multi-Panel Exploration', fontsize=16, fontweight='bold', y=1.01)
plt.show()

---
## 6. Seaborn Integration: Statistical Plots Made Easy

### 6.1 Relational and Categorical Plots

In [ ]:
df_tips = sns.load_dataset('tips')
df_penguins = sns.load_dataset('penguins').dropna()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sns.scatterplot(data=df_tips, x='total_bill', y='tip', hue='time', size='size',
                sizes=(20, 200), alpha=0.7, ax=axes[0, 0])
axes[0, 0].set_title('scatterplot: Tip vs Total Bill')

sns.lineplot(data=df_tips, x='size', y='tip', hue='sex',
             marker='o', ax=axes[0, 1], ci=None)
axes[0, 1].set_title('lineplot: Average Tip by Party Size')

sns.barplot(data=df_tips, x='day', y='total_bill', hue='sex',
            palette='Set2', ax=axes[0, 2])
axes[0, 2].set_title('barplot: Total Bill by Day and Sex')

sns.boxplot(data=df_penguins, x='species', y='body_mass_g', hue='sex',
            palette='Pastel1', ax=axes[1, 0])
axes[1, 0].set_title('boxplot: Body Mass by Species')

sns.violinplot(data=df_penguins, x='species', y='flipper_length_mm', hue='sex',
               split=True, palette='muted', ax=axes[1, 1])
axes[1, 1].set_title('violinplot: Flipper Length (split by sex)')

sns.kdeplot(data=df_penguins, x='flipper_length_mm', hue='species',
            fill=True, alpha=0.3, common_norm=False, ax=axes[1, 2])
axes[1, 2].set_title('kdeplot: Flipper Length Distributions')

plt.tight_layout()
plt.show()

### 6.2 Pairplot and Heatmap

In [ ]:
sns.pairplot(df_penguins, hue='species', palette='Set1', corner=True,
             diag_kind='kde', plot_kws=dict(alpha=0.6, s=25, edgecolor='white'))
plt.suptitle('pairplot: Penguin Measurements', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
df_iris_num = df_iris.drop(columns='species')
corr_iris = df_iris_num.corr()

mask = np.triu(np.ones_like(corr_iris, dtype=bool), k=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(corr_iris, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('heatmap: Iris Correlation Matrix')

sns.clustermap(corr_iris, annot=True, fmt='.2f', cmap='vlag', center=0,
               linewidths=1, figsize=(6, 5), cbar_pos=(0.05, 0.8, 0.03, 0.2))
axes[1].set_title('clustermap: Hierarchically Clustered', fontsize=11)

plt.tight_layout()
plt.show()

### 6.3 FacetGrid and relplot for Multi-Dimensional Data

In [ ]:
g = sns.FacetGrid(df_tips, col='time', row='sex', hue='day',
                  height=3.5, aspect=1.2, palette='Set2')
g.map(sns.scatterplot, 'total_bill', 'tip', alpha=0.7, edgecolor='white')
g.add_legend()
g.set_axis_labels('Total Bill ($)', 'Tip ($)')
g.fig.suptitle('FacetGrid: Tips by Time of Day and Gender', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
sns.relplot(data=df_penguins, x='bill_length_mm', y='bill_depth_mm',
            hue='species', style='island', size='body_mass_g', sizes=(20, 250),
            col='sex', kind='scatter', height=4, aspect=1.1,
            palette='deep')
plt.suptitle('relplot: Penguin Bill Dimensions by Island, Sex, and Species', y=1.05, fontsize=12)


### 6.4 Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

sns.kdeplot(data=df_penguins, x='body_mass_g', hue='species',
            fill=True, common_norm=False, alpha=0.35, palette='Set2',
            linewidth=2, ax=axes[0])
axes[0].set_title('KDE: Body Mass by Species')

sns.histplot(data=df_penguins, x='body_mass_g', hue='species',
             element='step', kde=True, palette='Set2', alpha=0.5, ax=axes[1])
axes[1].set_title('Histogram + KDE: Body Mass')

sns.ecdfplot(data=df_penguins, x='flipper_length_mm', hue='species',
             palette='Set2', linewidth=2, ax=axes[2])
axes[2].set_title('ECDF Plot: Flipper Length')

plt.tight_layout()
plt.show()

### 6.5 Seaborn Style Themes

In [ ]:
styles = ['darkgrid', 'whitegrid', 'dark', 'white', 'ticks']
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

x = np.linspace(0, 2 * np.pi, 100)
for ax, style in zip(axes, styles):
    with sns.axes_style(style):
        ax2 = fig.add_subplot(1, 5, axes.tolist().index(ax) + 1)
        ax2.plot(x, np.sin(x), color='#e41a1c', linewidth=2)
        ax2.plot(x, np.cos(x), color='#377eb8', linewidth=2)
        ax2.set_title(f'"{style}"', fontsize=10)
        ax2.set_xticks([])
        ax2.set_yticks([])

plt.suptitle('Seaborn Style Themes', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---
## 7. Plotly for Interactive Visualizations

In [ ]:
df_gapminder = px.data.gapminder()
data_2007 = df_gapminder[df_gapminder['year'] == 2007].copy()

fig = px.scatter(data_2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=60,
                 labels={'gdpPercap': 'GDP per Capita', 'lifeExp': 'Life Expectancy'},
                 title='Gapminder 2007: GDP vs Life Expectancy (hover for details)')
fig.update_layout(template='plotly_white', width=900, height=550)
fig.show()

In [ ]:
europe = data_2007[data_2007['continent'] == 'Europe'].sort_values('gdpPercap', ascending=False).head(10)

fig = px.bar(europe, x='country', y='gdpPercap', color='lifeExp',
             color_continuous_scale='Blues',
             labels={'gdpPercap': 'GDP per Capita', 'lifeExp': 'Life Expectancy'},
             title='Top 10 European Countries by GDP per Capita (2007)')
fig.update_layout(template='plotly_white', width=800, height=450)
fig.show()

In [ ]:
europe_trend = df_gapminder[df_gapminder['country'].isin(
    ['France', 'Germany', 'Italy', 'Spain', 'United Kingdom'])]

fig = px.line(europe_trend, x='year', y='lifeExp', color='country',
              markers=True,
              labels={'lifeExp': 'Life Expectancy', 'year': ''},
              title='Life Expectancy Trends in Western Europe')
fig.update_layout(template='plotly_white', width=800, height=450)
fig.show()

In [ ]:
fig = px.histogram(data_2007, x='lifeExp', color='continent',
                   nbins=25, marginal='box',
                   labels={'lifeExp': 'Life Expectancy', 'continent': 'Continent'},
                   title='Distribution of Life Expectancy by Continent (2007)')
fig.update_layout(template='plotly_white', width=850, height=500)
fig.show()

In [ ]:
fig = px.scatter_matrix(data_2007.dropna(),
                        dimensions=['lifeExp', 'gdpPercap', 'pop'],
                        color='continent',
                        title='Scatter Matrix: Life Expectancy, GDP, Population')
fig.update_layout(template='plotly_white', width=800, height=800)
fig.show()

In [ ]:
top_countries = data_2007.sort_values('pop', ascending=False).head(20)

fig = px.parallel_coordinates(top_countries,
                              dimensions=['lifeExp', 'gdpPercap', 'pop'],
                              color='lifeExp',
                              color_continuous_scale=px.colors.sequential.Viridis,
                              labels={'lifeExp': 'Life Exp', 'gdpPercap': 'GDP/cap', 'pop': 'Population'},
                              title='Parallel Coordinates: Top 20 Most Populous Countries')
fig.update_layout(template='plotly_white', width=900, height=500)
fig.show()

In [ ]:
fig = px.scatter_3d(data_2007, x='gdpPercap', y='lifeExp', z='pop',
                    color='continent', size='pop', size_max=50,
                    hover_name='country', log_x=True,
                    labels={'gdpPercap': 'GDP per Capita',
                            'lifeExp': 'Life Expectancy', 'pop': 'Population'},
                    title='3D Scatter: GDP, Life Expectancy, Population (2007)')
fig.update_layout(template='plotly_white', width=900, height=650)
fig.show()

---
## 8. Best Practices for Visualization

### Choosing the Right Chart

| Data Type | Chart Type |
|---|---|
| One categorical → one numeric | Bar chart, box plot |
| Two numeric | Scatter plot, line plot |
| Distribution | Histogram, KDE, ECDF, violin |
| Composition | Stacked bar, pie (rarely) |
| Correlation | Heatmap, pairplot, scatter matrix |
| Trend over time | Line chart, area chart |
| Multi-dimensional | FacetGrid, relplot, parallel coordinates |

### Colorblind-Friendly Palettes
- Use **viridis**, **plasma**, **inferno**, or **magma** instead of jet/rainbow
- For categorical: **Set2**, **Pastel1**, **Dark2** from ColorBrewer
- Avoid red-green combinations for accessibility

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

bad_cmap = plt.cm.jet
good_cmap = plt.cm.viridis
cb_cmap = plt.cm.cividis

data_grid = np.random.randn(20, 20)

axes[0].imshow(data_grid, cmap='jet', aspect='auto')
axes[0].set_title('jet: Perceptually Non-Uniform', fontsize=10)

axes[1].imshow(data_grid, cmap='viridis', aspect='auto')
axes[1].set_title('viridis: Perceptually Uniform', fontsize=10)

axes[2].imshow(data_grid, cmap='cividis', aspect='auto')
axes[2].set_title('cividis: Colorblind-Friendly', fontsize=10)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Color Map Comparison', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Tufte Principles Applied
1. **Maximize data-ink ratio** — remove non-data ink (excessive borders, 3D effects, backgrounds)
2. **Avoid chartjunk** — no gratuitous textures, shadows, or pseudo-3D
3. **Use small multiples** — FacetGrid for comparing across categories
4. **Show the data** — avoid distorting what the data say

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

data = [np.random.normal(0, std, 50) for std in range(1, 5)]

axes[0].violinplot(data, showmedians=True)
axes[0].set_xticks([1, 2, 3, 4])
axes[0].set_xticklabels(['A', 'B', 'C', 'D'])
axes[0].set_title('Before: Heavy Borders, Grid Overload', fontsize=10)
axes[0].grid(True, alpha=0.8)

vp = axes[1].violinplot(data, showmedians=True)
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xticklabels(['A', 'B', 'C', 'D'])
axes[1].set_title('After: Clean, Minimal Ink', fontsize=10)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].spines['left'].set_color('#cccccc')
axes[1].spines['bottom'].set_color('#cccccc')
axes[1].tick_params(colors='#666666')
for body in vp['bodies']:
    body.set_facecolor('#2c7bb6')
    body.set_alpha(0.5)

plt.tight_layout()
plt.show()

---
## 9. Visualization for Machine Learning

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import label_binarize

X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_redundant=3, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_score = clf.predict_proba(X_test)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True,
            ax=axes[0, 0], cbar_kws={'shrink': 0.75},
            xticklabels=['Class 0', 'Class 1', 'Class 2'],
            yticklabels=['Class 0', 'Class 1', 'Class 2'])
axes[0, 0].set_title('Confusion Matrix Heatmap', fontsize=12)
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

y_bin = label_binarize(y_test, classes=[0, 1, 2])
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0, 1].plot(fpr, tpr, color=colors[i], lw=2,
                    label=f'Class {i} (AUC = {roc_auc:.2f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0, 1].set_title('ROC Curves (One-vs-Rest)', fontsize=12)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend(fontsize=9)
axes[0, 1].set_xlim(-0.02, 1.02)
axes[0, 1].set_ylim(-0.02, 1.02)

importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
axes[1, 0].barh(range(10), importances[indices][::-1], color='#1b9e77', edgecolor='white')
axes[1, 0].set_yticks(range(10))
axes[1, 0].set_yticklabels([f'Feature {i}' for i in indices][::-1])
axes[1, 0].set_title('Feature Importance', fontsize=12)
axes[1, 0].set_xlabel('Importance')
axes[1, 0].spines['top'].set_visible(False)
axes[1, 0].spines['right'].set_visible(False)

train_sizes, train_scores, test_scores = learning_curve(
    clf, X, y, cv=5, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy')
train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_std = np.std(test_scores, axis=1)

axes[1, 1].plot(train_sizes, train_mean, 'o-', color='#e41a1c', label='Training Score')
axes[1, 1].plot(train_sizes, test_mean, 'o-', color='#377eb8', label='Cross-Validation Score')
axes[1, 1].fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                        alpha=0.15, color='#e41a1c')
axes[1, 1].fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                        alpha=0.15, color='#377eb8')
axes[1, 1].set_title('Learning Curves', fontsize=12)
axes[1, 1].set_xlabel('Training Examples')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend(loc='lower right', fontsize=9)
axes[1, 1].set_ylim(0.5, 1.02)

plt.suptitle('ML Visualization Toolkit', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Saving Figures

In [ ]:
import os

fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(0, 2 * np.pi, 100)
ax.plot(x, np.sin(x), linewidth=2, label='sin(x)')
ax.plot(x, np.cos(x), linewidth=2, label='cos(x)')
ax.set_title('Saving Figures Demo', fontsize=13)
ax.legend()

os.makedirs('figures', exist_ok=True)

fig.savefig('figures/demo_high_dpi.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
fig.savefig('figures/demo_vector.svg', format='svg', bbox_inches='tight')
fig.savefig('figures/demo_pdf.pdf', format='pdf', bbox_inches='tight')
fig.savefig('figures/demo_transparent.png', dpi=150, transparent=True, bbox_inches='tight')

print('Saved files:')
for f in ['figures/demo_high_dpi.png', 'figures/demo_vector.svg',
          'figures/demo_pdf.pdf', 'figures/demo_transparent.png']:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

plt.show()

print('\nKey savefig parameters:')
print('  dpi=300         - high resolution for publication')
print('  bbox_inches="tight" - crop whitespace')
print('  transparent=True - transparent background')
print('  facecolor/edgecolor - background color')

---
## 11. Pandas Built-In Plotting

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

df_tips.groupby('day')['total_bill'].mean().plot(kind='bar', ax=axes[0, 0],
    color=['#e41a1c', '#377eb8', '#4daf4a', '#984ea3'], edgecolor='white')
axes[0, 0].set_title('.plot(kind="bar")', fontsize=10)
axes[0, 0].set_ylabel('Mean Total Bill ($)')

df_tips['total_bill'].plot(kind='hist', ax=axes[0, 1], bins=25,
                            color='#1b9e77', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('.plot(kind="hist")', fontsize=10)
axes[0, 1].set_xlabel('Total Bill ($)')

df_tips.boxplot(column='tip', by='sex', ax=axes[1, 0])
axes[1, 0].set_title('.boxplot()', fontsize=10)
axes[1, 0].set_ylabel('Tip ($)')

df_iris.drop(columns='species').plot(kind='line', ax=axes[1, 1], alpha=0.8)
axes[1, 1].set_title('.plot(kind="line") — All Iris Features', fontsize=10)
axes[1, 1].set_xlabel('Row Index')
axes[1, 1].legend(fontsize=8)

plt.suptitle('Pandas Built-In Plotting Methods', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Practice Exercises

### Exercise 1: Custom Line Plot
Load `sns.load_dataset('flights')`. Create a line plot showing passengers per month over years. Use a different color for each month, add markers, and customize the legend to appear outside the plot area.

**Hint:** `pivot_table(index='year', columns='month', values='passengers')` first.

### Exercise 2: Seaborn Pairplot Analysis
Load `sns.load_dataset('diamonds')`. Create:
- A `sns.pairplot()` of `carat`, `depth`, `table`, `price` colored by `cut`
- A `sns.heatmap()` of the correlation matrix for numeric columns
- A `sns.catplot()` showing price distribution by cut (hint: use `kind='box'`)

### Exercise 3: Interactive Plotly Dashboard
Using `px.data().gapminder()`, create an animated scatter plot (`px.scatter()`) with `animation_frame='year'`. Map GDP (x), life expectancy (y), population (size), and continent (color). Add animation controls and a play button.

### Exercise 4: ML Visualization Pipeline
Train a logistic regression on `sns.load_dataset('penguins')` to predict species from bill_length_mm and bill_depth_mm. Create:
- A scatter plot of the training data with decision boundaries
- A confusion matrix heatmap
- A classification report as a table

**Hint:** Use `from sklearn.linear_model import LogisticRegression`

### Exercise 5: Complex GridSpec Layout
Using `GridSpec`, create a 3×3 layout containing:
- Top-left (2×2): scatter plot of random data colored by cluster
- Top-right (2×1): histogram of feature A
- Middle-right (1×1): box plot of feature B by cluster
- Bottom (3×1): a line plot showing the centroids across iterations
- Bottom-right area: leave empty but add a text annotation summarizing key statistics